In [ ]:
# ============================================================
# ResNet50 Transfer Learning + Fine-Tuning (Keras / TensorFlow)
# ============================================================
import os, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet import preprocess_input  # modo "caffe": BGR + mean

In [ ]:
# -------------------------
# 0) Configuración general
# -------------------------
SEED = 42
IMG_SIZE = (224, 224)
BATCH = 32
DATA_DIR_TRAIN = "data/train"   # <-- cambia a tu ruta
DATA_DIR_VAL   = "data/val"     # <-- cambia a tu ruta
BEST_WEIGHTS   = "best_resnet50_finetune.keras"

tf.keras.utils.set_random_seed(SEED)
AUTOTUNE = tf.data.AUTOTUNE

In [ ]:
# -----------------------------------
# 1) Carga de datos desde directorios
# -----------------------------------
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR_TRAIN,
    labels="inferred",
    label_mode="categorical",
    image_size=IMG_SIZE,
    batch_size=BATCH,
    shuffle=True,
    seed=SEED,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR_VAL,
    labels="inferred",
    label_mode="categorical",
    image_size=IMG_SIZE,
    batch_size=BATCH,
    shuffle=False
)

num_classes = train_ds.element_spec[1].shape[-1]

# Rendimiento
train_ds = train_ds.cache().prefetch(AUTOTUNE)
val_ds   = val_ds.cache().prefetch(AUTOTUNE)

In [ ]:
# --------------------------------------------
# 2) Aumentación + preprocesamiento ResNet50
# --------------------------------------------
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
], name="augment")

def to_255(x):  # volver a [0,255] tras augment en [0,1]
    return x * 255.0

preproc = keras.Sequential([
    layers.Rescaling(1./255),           # [0,255] -> [0,1]
    data_augmentation,                  # augment en [0,1]
    layers.Lambda(to_255),              # [0,1] -> [0,255]
    layers.Lambda(preprocess_input),    # ResNet50 "caffe": BGR + mean subtraction
], name="preproc")

In [ ]:
# -------------------------------
# 3) Construcción del modelo base
# -------------------------------
base = ResNet50(
    include_top=False,
    weights="imagenet",
    input_shape=IMG_SIZE + (3,),
    pooling="avg"   # Global Average Pooling
)

# Fase 1: congelar la base
base.trainable = False

inputs = keras.Input(shape=IMG_SIZE + (3,))
x = preproc(inputs)
x = base(x, training=False)             # BN en modo inferencia al congelar
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)

model = keras.Model(inputs, outputs, name="resnet50_transfer_finetune")

In [ ]:
# ----------------------
# 4) Entrenamiento (Fase 1: cabeza)
# ----------------------
initial_lr = 1e-3
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=initial_lr),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks = [
    keras.callbacks.ModelCheckpoint(
        BEST_WEIGHTS, monitor="val_accuracy", save_best_only=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=5, restore_best_weights=True
    )
]

EPOCHS_HEAD = 8
history_head = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_HEAD,
    callbacks=callbacks
)

In [ ]:
# ---------------------------------------
# 5) Fine-tuning (Fase 2): desbloquear conv5_*
# ---------------------------------------
# Descongelamos el último "stage" de ResNet50 (conv5_block*)
for layer in base.layers:
    if layer.name.startswith("conv5_"):  # conv5_block1_*, conv5_block2_*, conv5_block3_*
        layer.trainable = True
    else:
        layer.trainable = False

# Recompilar con LR más bajo
fine_tune_lr = 1e-5
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=fine_tune_lr),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

EPOCHS_FT = 15
history_ft = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FT,
    callbacks=callbacks
)

# Cargar mejores pesos
model = keras.models.load_model(BEST_WEIGHTS)


In [ ]:
# ----------------------
# 6) Evaluación y guardado
# ----------------------
val_loss, val_acc = model.evaluate(val_ds, verbose=0)
print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

model.save("resnet50_finetuned_savedmodel")  # SavedModel
model.save("resnet50_finetuned.keras")       # Formato Keras